# 03 - Modelagem, IA e Sistemas Inteligentes

Notebook responsável pelas análises avançadas, modelagem estatística, inteligência artificial, sistemas de recomendação e experimentos de IA generativa aplicados aos dados de compras.

Objetivos:
- Market Basket Analysis
- Sistemas de Recomendação
- Embeddings de Produtos
- IA Generativa
- Insights Automáticos
- Modelagem Temporal

In [1]:
# !pip install -r requirements.txt

In [2]:
# IMPORTS
import time
from pathlib import Path
import pandas as pd
import warnings

from src.core.tictoc import tictoc
from src.core.config_loader import ConfigLoader
from src.core.paths import ROOT_DIR
from src.core.logger import setup_logger

# CONFIGURAÇÃO
warnings.filterwarnings('ignore')
logger = setup_logger(__name__)
tic_geral = time.time()

config = ConfigLoader(
    "config/settings.yaml"
)

## 1. Carregamento dos Dados

Nesta etapa é realizado o carregamento da base analítica consolidada utilizada nos experimentos de modelagem estatística, inteligência artificial, sistemas de recomendação e análises avançadas de comportamento de consumo.

In [3]:
tic = time.time()

In [4]:
data_path = config.get(
    "paths.data"
)
df = pd.read_excel(data_path)
print(
    f"Linhas: {df.shape[0]}"
)
print(
    f"Colunas: {df.shape[1]}"
)
df.head()

Linhas: 3430
Colunas: 25


,chave_anonimizada,data_hora,mes_ano,dia_semana,feriado,estacao_ano,temperatura_max,temperatura_min,temperatura_media,chuva_mm,...,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,categoria_produto,produto,unidade,quantidade,preco_unitario,preco_total
0,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,10176020001,CEREAL,CEREAL NESCAU 770G,UNIDADE,1.0,19.90,19.90
1,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11421800001,MERCEARIA,BEB VERY 1250G SALAD,UNIDADE,1.0,7.70,7.70
2,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,18730001,MERCEARIA,ARR T JOAO LF T1 5kg,UNIDADE,1.0,25.90,25.90
3,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,65420001,BEBIDA NAO ALCOOLICA,BEB NESQUIK 200ML MO,UNIDADE,10.0,2.89,28.90
4,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,10566170001,MOLHO,EXT ELEF TP 140G,UNIDADE,2.0,2.59,5.18


In [5]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 1 | Tempo: (00:00:02) [19/05/2026 13:26:15 - [19/05/2026 13:26:17]


### 2. Entendimento Inicial dos Dados

In [6]:
tic = time.time()

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3430 entries, 0 to 3429
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   chave_anonimizada     3430 non-null   object        
 1   data_hora             3430 non-null   datetime64[ns]
 2   mes_ano               3430 non-null   object        
 3   dia_semana            3430 non-null   object        
 4   feriado               3430 non-null   object        
 5   estacao_ano           3430 non-null   object        
 6   temperatura_max       3430 non-null   float64       
 7   temperatura_min       3430 non-null   float64       
 8   temperatura_media     3430 non-null   float64       
 9   chuva_mm              3430 non-null   float64       
 10  cat_temperatura       3430 non-null   object        
 11  dia_chuvoso           3430 non-null   bool          
 12  periodo_dia           3430 non-null   object        
 13  cnpj              

In [8]:
df.isna().mean().sort_values(
    ascending=False
)

chave_anonimizada       0.0
data_hora               0.0
mes_ano                 0.0
dia_semana              0.0
feriado                 0.0
estacao_ano             0.0
temperatura_max         0.0
temperatura_min         0.0
temperatura_media       0.0
chuva_mm                0.0
cat_temperatura         0.0
dia_chuvoso             0.0
periodo_dia             0.0
cnpj                    0.0
supermercado            0.0
qtd_total_nota          0.0
valor_total_nota        0.0
valor_total_tributos    0.0
cod_produto             0.0
categoria_produto       0.0
produto                 0.0
unidade                 0.0
quantidade              0.0
preco_unitario          0.0
preco_total             0.0
dtype: float64

In [9]:
df.nunique().sort_values(
    ascending=False
)

produto                 1087
cod_produto             1087
preco_total              694
preco_unitario           445
quantidade               218
data_hora                177
chave_anonimizada        177
valor_total_nota         175
valor_total_tributos     169
temperatura_max           90
temperatura_min           90
temperatura_media         87
chuva_mm                  73
qtd_total_nota            57
mes_ano                   46
categoria_produto         20
unidade                   10
dia_semana                 7
cnpj                       6
periodo_dia                4
estacao_ano                4
cat_temperatura            4
supermercado               4
feriado                    2
dia_chuvoso                2
dtype: int64

In [10]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 2 | Tempo: (00:00:00) [19/05/2026 13:26:17 - [19/05/2026 13:26:17]


### 3. Feature Engineering

In [33]:
tic = time.time()

In [34]:
toc = time.time()
print(f"\n\033[33mEtapa 3 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 3 | Tempo: (00:00:00) [19/05/2026 13:11:58 - [19/05/2026 13:11:58]


### 4. Market Basket Analysis

Nesta etapa são identificadas relações entre produtos e categorias frequentemente comprados em conjunto utilizando técnicas de associação.

In [35]:
tic = time.time()

In [36]:
basket = (

    df

    .groupby([
        'chave_anonimizada',
        'categoria_produto'
    ])

    .size()

    .unstack(fill_value=0)
)

In [37]:
basket = (

    basket > 0

).astype(int)

basket.head()

categoria_produto,ACOUGUE,BEBIDA ALCOOLICA,BEBIDA NAO ALCOOLICA,BISCOITO E SNACK,BOMBONIERE,CEREAL,CONDIMENTO,CONGELADO,COSMETICO,HIGIENE E LIMPEZA,HORTIFRUTI,LATARIA E CONSERVA,LATICINIO,MASSA,MERCEARIA,MOLHO,OUTROS,PADARIA E FRIOS,PET,UTILIDADES
chave_anonimizada,,,,,,,,,,,,,,,,,,,,
00448fcaa702aee48b5d955a696d63ac108f96e24cd26046f8e42503aec4966b,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,0,1,0,0
038ae8d0a63c6c9a4b94d01b536029f3d60255be5d670dbd9849972dd7bd7b3a,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0481fe1cb68c045ad74b2be30b5e3a23ab8c56f85532a31fae035fd0238c7b3f,0,0,1,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0
04f9752cf3cc41294b2df6a0b2212f30e4194850c2f7b45adacd4eb5b8621a67,1,1,1,0,1,0,1,1,0,1,1,0,0,1,1,1,0,1,0,1
05c70ffd03ac5966153019a654c38e64a5a3f0e528adda98d89ab9b7bcaf3f0c,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


In [38]:
from mlxtend.frequent_patterns import (
    fpgrowth,
    association_rules
)

ModuleNotFoundError: No module named 'mlxtend'

In [ ]:
frequent_items = fpgrowth(

    basket,

    min_support=0.01,

    use_colnames=True
)

In [ ]:
rules = association_rules(

    frequent_items,

    metric='lift',

    min_threshold=1
)

In [ ]:
rules[
    [
        'antecedents',
        'consequents',
        'support',
        'confidence',
        'lift'
    ]
]

.sort_values(
    'lift',
    ascending=False
)

.head(20)

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 4 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 5. NLP & Embeddings

Nesta etapa são gerados embeddings semânticos dos produtos para análises de similaridade, agrupamento e recomendação.

In [ ]:
tic = time.time()

In [ ]:
df['produto_clean']

In [ ]:
from sentence_transformers import (
    SentenceTransformer
)

In [ ]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

In [ ]:
embeddings = model.encode(
    df['produto'].unique()
)

In [ ]:
cosine_similarity(...)

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 5 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 6. IA Generativa e Insights Automáticos

Nesta etapa são exploradas abordagens de LLMs para geração automática de insights, explicações estatísticas e interpretação dos padrões de consumo.

In [ ]:
tic = time.time()

In [ ]:
def generate_insight(...):

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 6 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 8. Próximos Passos

- Sistemas de recomendação
- Forecast temporal
- Detecção de anomalias
- RAG sobre dados financeiros
- IA contextual com clima e inflação

## 3. Tempo Decorrido

In [ ]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")